# Pipeline Interactivo de Clasificación de Flores

## Bienvenida

En este notebook vas a **experimentar en tiempo real** cómo funciona un pipeline completo de visión por computadora. 

Desde la Unidad 1 has aprendido a leer imágenes y procesar píxeles. En la Unidad 2 exploraste cómo realzar y mejorar imágenes. En la Unidad 3 descubriste técnicas de segmentación. En la Unidad 4 aprendiste a extraer características (color, textura, forma) y a usarlas para entrenar clasificadores. Ahora, en esta actividad interactiva, vas a **ver cómo todos esos pasos se conectan en un pipeline real**.

Específicamente, vas a descubrir:

1. **¿Qué características capturan mejor la identidad de una flor?** Vas a probar si usar solo color, solo textura, solo forma, o una combinación de todas.
2. **¿Cuál es el algoritmo más inteligente para esta tarea?** Compararás cuatro algoritmos: Regresión Logística, SVM, Bosques Aleatorios, y k-Vecinos.
3. **¿Cómo cambia el desempeño cuando ajustas los parámetros?** Cambiarás valores como la rigidez (C en SVM), el número de árboles en Random Forest, o el número de vecinos en kNN, y verás cómo impacta.

## Cómo usar este notebook

1. **Ejecuta las celdas en orden** — las primeras descargan datos y preparan las herramientas que necesitamos
2. **Llega hasta la sección "Panel de Control"** — ahí es donde comienza la experimentación
3. **Observa cómo cambian los resultados** cada vez que modificas características, algoritmo o parámetros
4. **Discute con tus compañeros** — ¿por qué funciona mejor una combinación que otra? ¿Qué aprendiste?

---

In [ ]:
# PASO 0: Preparar el entorno
# Esta celda descarga y configura todo lo que necesitamos. Solo necesitas ejecutarla una vez.

import sys
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('✓ Ejecutando en Colab' if IN_COLAB else '✓ Ejecutando localmente')

# Instalar dependencias (si es necesario)
if IN_COLAB:
    !pip -q install -U scikit-image ipywidgets

# Importar librerías
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Configurar widgets (sliders y botones interactivos)
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

import ipywidgets as widgets

print('✓ Todo listo. Puedes pasar a la siguiente celda.')

## PASO 1: Cargar el dataset de flores

Vamos a usar un dataset clásico de flores: **5 tipos diferentes** (daisies, dandelions, roses, sunflowers, tulips). Cada imagen será analizada de diferentes formas para extraer sus características.

Recuerda de la Unidad 1: leer una imagen es el primer paso. Aquí usamos TensorFlow para descargar automáticamente un conjunto de imágenes de flores. Cada una será procesada con los descriptores que aprendiste en la Unidad 4: color (media y desviación estándar RGB), textura (GLCM con propiedades estadísticas), y forma (momentos de Hu). Luego, estos descriptores servirán como entrada a los algoritmos de clasificación.

Ejecuta la siguiente celda para descargar el dataset.

In [ ]:
# 📥 Descargar y explorar el dataset de flores

import tensorflow as tf
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'

print('Descargando ~220 MB de imágenes de flores...')
# Descargar y extraer
path_to_zip = tf.keras.utils.get_file('flower_photos.tgz', origin=DATA_URL, extract=True)
data_root = Path(path_to_zip).parent / 'flower_photos'

# Si se extrajo bajo otra carpeta (por ejemplo, flower_photos_extracted)
if not data_root.exists() or not any(data_root.iterdir()):
    posibles_rutas = list(Path(path_to_zip).parent.glob('*flower_photos*'))
    for p in posibles_rutas:
        if p.is_dir():
            data_root = p
            break

# Si dentro de la carpeta extraída hay OTRA subcarpeta llamada 'flower_photos'
if (data_root / 'flower_photos').exists() and (data_root / 'flower_photos').is_dir():
    data_root = data_root / 'flower_photos'

print(f'✓ Dataset guardado en: {data_root}')

if not data_root.exists():
    raise FileNotFoundError(f"No se encontró la carpeta extraída en: {data_root}")

# Identificar las clases (tipos de flores)
classes = sorted([p.name for p in data_root.iterdir() if p.is_dir() and p.name != 'LICENSE.txt'])
print(f'✓ Clases encontradas: {classes}')

# Recolectar todas las rutas de imágenes
all_paths = []
all_labels = []
for i, cls in enumerate(classes):
    for img_path in (data_root/cls).glob('*.jpg'):
        all_paths.append(str(img_path))
        all_labels.append(i)

all_paths = np.array(all_paths)
all_labels = np.array(all_labels)
print(f'✓ Total de imágenes: {len(all_paths)}')

# Mostrar ejemplos visuales
print('\nMostrando 9 flores aleatorias del dataset...')
if len(all_paths) > 0:
    idx = np.random.choice(len(all_paths), size=9, replace=False)
    plt.figure(figsize=(8,8))
    for j,k in enumerate(idx):
        img = tf.keras.utils.load_img(all_paths[k], target_size=(160,160))
        plt.subplot(3,3,j+1)
        plt.imshow(img)
        plt.title(classes[all_labels[k]], fontsize=10)
        plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("¡Error! No se encontraron imágenes en el directorio especificado.")

## PASO 2: Configurar los parámetros del experimento

Aquí definimos una **configuración base** que usaremos para el primer experimento. Más adelante podrás cambiar cada valor y ver cómo impacta en los resultados.

Recuerda de la Unidad 4: al entrenar un clasificador necesitas decidir qué descriptor usar (color, textura, forma, o todos), qué algoritmo elegir (Regresión Logística, SVM, Random Forest, kNN), y qué parámetros ajustar para cada algoritmo. Esta configuración base es el punto de partida. Los valores que ves aquí (120 imágenes por clase, tamaño 96x96, fusión de características, SVM con C=3.0) se eligieron para que sea rápido de ejecutar en clase sin perder información importante.

Siéntete libre de cambiar cualquier valor cuando llegues a la sección interactiva.

In [ ]:
# PASO 2: Configuración por defecto (que puedes cambiar después)

BASE_CONFIG = {
    # Datos: mantener bajo para que sea rápido en clase
    'max_per_class': 120,      # Máximo de imágenes por tipo de flor
    'img_size': 96,            # Tamaño de cada imagen (96x96 píxeles)

    # Características: ¿Cómo vamos a "ver" las flores?
    'feature_set': 'fusion',   # color | textura | forma | fusion

    # Modelo: ¿Qué algoritmo enseñamos?
    'model': 'svm',            # logreg | svm | rf | knn
    'svm_c': 3.0,              # Parámetro C de SVM (rigidez)
    'svm_gamma': 'scale',      # Parámetro gamma de SVM
    'rf_trees': 250,           # Número de árboles en Random Forest
    'knn_k': 7,                # Número de vecinos en k-NN

    # División: cómo repartimos datos de entrenamiento y prueba
    'test_size': 0.25,         # 25% para probar, 75% para entrenar
    'random_state': 42,        # Semilla para reproducibilidad
}

print('✓ Configuración base cargada')
print(f'  - Características: {BASE_CONFIG["feature_set"]}')
print(f'  - Modelo: {BASE_CONFIG["model"]}')
print(f'  - Imágenes por clase: {BASE_CONFIG["max_per_class"]}')

## PASO 3: Las funciones de extracción de características

Esta sección define los **tres descriptores estándar** del curso que aprendiste en la Unidad 4.

Recordatorio rápido:
- **Color**: Extraemos la media, desviación estándar y sesgo de cada canal RGB. Funciona bien cuando las flores tienen colores muy distintos (una rosa roja será muy diferente de un girasol amarillo).
- **Textura (GLCM)**: Calculamos la matriz de co-ocurrencia de grises en 3 distancias (1, 3, 5) y 4 ángulos (0°, 45°, 90°, 135°), extrayendo 6 propiedades por cada combinación (contrast, dissimilarity, homogeneity, ASM, energy, correlation). Total: 72 variables. Funciona bien cuando la textura de los pétalos es característica (pétalos lisos vs. rugosos).
- **Forma (Momentos de Hu)**: Extraemos siete momentos invariantes de la silueta. Funciona bien cuando la forma es muy diferente entre flores.

Cada descriptor tiene fortalezas diferentes. Tu tarea será descubrir cuál (o cuál combinación) funciona mejor para el reconocimiento de flores.

Resumen de dimensiones:
- Color: 9 variables
- Textura: 72 variables
- Forma: 7 variables
- Fusión: 88 variables total



In [ ]:
# PASO 3: Funciones de extracción de características (estándar del curso Unidad 4)

import cv2
from skimage.feature import graycomatrix, graycoprops
from scipy import stats


def load_rgb(path, img_size):
    """Cargar imagen BGR con OpenCV y convertir a RGB"""
    bgr = cv2.imread(path)
    if bgr is None:
        raise ValueError(f'No se pudo leer: {path}')
    bgr = cv2.resize(bgr, (img_size, img_size), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb


def extract_color_features(image_rgb):
    """Descriptor de COLOR: Media, desv. est. y sesgo de RGB
    
    Captura: tonalidades, variabilidad y asimetría de color
    Útil para: Flores con colores muy distintos
    Salida: 9 valores (3 medias + 3 desv.est. + 3 sesgo RGB)
    Nota: Idéntico a Unidad 4
    """
    color_values = image_rgb.astype(np.float32)
    mean_color = color_values.mean(axis=0).mean(axis=0)    # Media por cada canal
    std_color = color_values.std(axis=0).mean(axis=0)      # Desv.est. promediada
    skew_color = np.array([
        stats.skew(color_values[:,:,i].flatten()) 
        for i in range(3)
    ])  # Sesgo por cada canal
    return np.concatenate([mean_color, std_color, skew_color]).astype(np.float32)


def extract_texture_features(image_rgb):
    """Descriptor de TEXTURA: GLCM (matriz de co-ocurrencia)
    
    Captura: patrones locales, rugosidad, regularidad
    Proceso:
      1. Convertir a escala de grises, cuantizar a 32 niveles
      2. Calcular GLCM con distancias [1,3,5] y 4 ángulos (0°, 45°, 90°, 135°)
      3. Extraer 6 propiedades por cada (distancia, ángulo):
         contrast, dissimilarity, homogeneity, ASM, energy, correlation
    Salida: 72 valores (6 propiedades × 3 distancias × 4 ángulos)
    Nota: Idéntico a Unidad 4
    """
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    # Cuantizar a 32 niveles para GLCM
    quantized = (gray.astype(np.uint16) * 32 // 256).astype(np.uint8)
    quantized = np.clip(quantized, 0, 31)
    
    # GLCM: distancias [1,3,5], ángulos [0°, 45°, 90°, 135°]
    glcm = graycomatrix(
        quantized,
        distances=[1, 3, 5],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=32,
        symmetric=True,
        normed=True
    )
    
    # Extraer 6 propiedades × 3 distancias × 4 ángulos = 72 valores
    props = ['contrast', 'dissimilarity', 'homogeneity', 'ASM', 'energy', 'correlation']
    texture_features = []
    
    for prop in props:
        vals = graycoprops(glcm, prop)  # Shape: (n_distances, n_angles)
        for i in range(vals.shape[0]):  # Para cada distancia
            for j in range(vals.shape[1]):  # Para cada ángulo
                texture_features.append(vals[i, j])
    
    return np.array(texture_features, dtype=np.float32)


def extract_shape_features(image_rgb):
    """Descriptor de FORMA: Momentos de Hu
    
    Captura: silueta y geometría (invariante a rotación y escala)
    Proceso:
      1. Convertir a escala de grises
      2. Calcular momentos de la imagen
      3. Extraer 7 momentos de Hu con transformación log
    Útil para: Flores con formas muy diferentes
    Salida: 7 valores transformados en escala log
    Nota: Idéntico a Unidad 4
    """
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    # Usar la imagen como máscara (todos los píxeles)
    moments = cv2.moments((gray > 0).astype(np.uint8) * 255)
    hu = cv2.HuMoments(moments).flatten()
    # Transformación log para estabilidad numérica
    shape_features = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    return shape_features.astype(np.float32)


def extract_features(paths, cfg):
    """Extraer características de un lote de imágenes según la configuración
    
    Combina los descriptores seleccionados:
    - 'color': solo color (9 valores: 3 media + 3 std + 3 skew)
    - 'textura': solo textura (72 valores: GLCM 6 props × 3 dist × 4 ang)
    - 'forma': solo forma (7 valores: momentos de Hu)
    - 'fusion': todos juntos (9+72+7=88 valores)
    """
    X = []
    for p in paths:
        rgb = load_rgb(p, cfg['img_size'])
        feats = []
        fs = cfg['feature_set']
        
        # Agregar los descriptores seleccionados
        if fs in ('color', 'fusion'):
            feats.append(extract_color_features(rgb))
        if fs in ('textura', 'fusion'):
            feats.append(extract_texture_features(rgb))
        if fs in ('forma', 'fusion'):
            feats.append(extract_shape_features(rgb))

        X.append(np.concatenate(feats))

    return np.vstack(X).astype(np.float32)

print('✓ Funciones de características del curso Unidad 4 cargadas')
print('  - Color: 9 variables')
print('  - Textura (GLCM): 72 variables (6 props × 3 dist × 4 ang)')
print('  - Forma (Hu): 7 variables')
print('  - Fusión: 88 variables total')

## PASO 4: Los cuatro algoritmos de clasificación

Recordatorio de la Unidad 4: una vez que tenemos características, necesitamos un clasificador que aprenda a separarlas por clase.

La Unidad 4 te presentó varios algoritmos supervisados. Aquí usamos cuatro:
- **Regresión Logística**: Es simple y rápida. Funciona bien si las clases son separables de forma lineal (con una línea recta o un plano).
- **SVM (Support Vector Machine)**: Es potente y versátil. Puede encontrar límites complejos y no lineales entre clases usando el parámetro C (que controla la rigidez del modelo).
- **Random Forest**: Es robusto porque usa muchos árboles de decisión. Generalmente funciona bien sin ajuste fino, pero puedes cambiar el número de árboles.
- **k-Vecinos (kNN)**: Es simple y visual: clasifica cada imagen según la mayoría de sus k vecinos más cercanos. El parámetro k controla cuántos vecinos consultar.

El hecho interesante: **con los mismos datos y características, algoritmos diferentes pueden dar resultados distintos**. Tu trabajo es explorar cuál es mejor para este problema.



In [ ]:
# PASO 4: Función para construir el modelo (con su algoritmo y configuración)

def build_estimator(cfg):
    """Crear un pipeline con escalado + algoritmo de clasificación"""
    model = cfg['model']

    if model == 'logreg':
        clf = LogisticRegression(max_iter=2000, n_jobs=None)
        return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

    if model == 'svm':
        clf = SVC(C=float(cfg['svm_c']), gamma=cfg['svm_gamma'], kernel='rbf')
        return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

    if model == 'rf':
        clf = RandomForestClassifier(n_estimators=int(cfg['rf_trees']), 
                                     random_state=cfg['random_state'], n_jobs=-1)
        return Pipeline([('clf', clf)])

    if model == 'knn':
        clf = KNeighborsClassifier(n_neighbors=int(cfg['knn_k']))
        return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

    raise ValueError(f"Modelo no soportado: {model}")

print('✓ Función de construcción de modelo cargada')

## PASO 5: La función que ejecuta un experimento completo

Recordatorio del flujo completo (pipeline):
1. **Muestreo balanceado**: Selecciona la misma cantidad de imágenes de cada clase (para evitar sesgos).
2. **Extracción de características**: Aplica el descriptor elegido (color, textura, forma, o fusión) a cada imagen.
3. **División en entrenamiento y prueba**: Separa 75% para entrenar y 25% para evaluar (recuerda: nunca entrenas con datos de prueba).
4. **Entrenamiento**: El algoritmo aprende a partir de las características de entrenamiento.
5. **Evaluación**: Predice las flores de prueba y compara con la verdad.
6. **Visualización**: Muestra la matriz de confusión (qué flores se confunden entre sí) y las métricas (precisión, recall, F1).

La celda siguiente implementa todo esto en la función `run_experiment()`. Cuando presiones el botón "Ejecutar", se llamará a esta función con tu configuración elegida.



In [ ]:
# PASO 5: Función para ejecutar un experimento completo

from sklearn.metrics import classification_report

_feature_cache = {}  # Cache para evitar recalcular características


def sample_balanced(paths, labels, max_per_class, random_state=42):
    """Seleccionar imágenes balanceadas de cada clase"""
    rng = np.random.default_rng(random_state)
    keep_idx = []
    for cls in np.unique(labels):
        idx = np.where(labels == cls)[0]
        k = min(max_per_class, len(idx))
        keep_idx.append(rng.choice(idx, size=k, replace=False))
    keep_idx = np.concatenate(keep_idx)
    rng.shuffle(keep_idx)
    return paths[keep_idx], labels[keep_idx]


def filter_by_classes(paths, labels, selected_classes):
    """Filtrar imágenes solo de las clases seleccionadas
    
    Recibe:
      - paths: array con rutas de imágenes
      - labels: array con etiquetas (índice de clase)
      - selected_classes: lista de nombres de clases a mantener
    
    Devuelve:
      - paths filtradas
      - labels filtradas (re-indexadas: 0, 1, 2, ... para las clases seleccionadas)
      - mapping: mapeo de etiqueta vieja -> nueva
    """
    # Encontrar índices de las clases seleccionadas
    class_to_idx = {cls: i for i, cls in enumerate(classes)}
    selected_indices = [class_to_idx[cls] for cls in selected_classes]
    
    # Filtrar
    mask = np.isin(labels, selected_indices)
    filtered_paths = paths[mask]
    filtered_labels = labels[mask]
    
    # Re-indexar etiquetas (mapear indices viejos a nuevos)
    mapping = {old_idx: new_idx for new_idx, old_idx in enumerate(selected_indices)}
    new_labels = np.array([mapping[label] for label in filtered_labels])
    
    return filtered_paths, new_labels, selected_classes


def run_experiment(cfg, selected_classes=None):
    """Ejecutar un experimento completo: extracción → entrenamiento → evaluación
    
    Si selected_classes es None, usa todas las clases
    """
    if selected_classes is None:
        selected_classes = classes
        paths_filtered = all_paths
        labels_filtered = all_labels
        selected_class_names = classes
    else:
        paths_filtered, labels_filtered, selected_class_names = filter_by_classes(
            all_paths, all_labels, selected_classes
        )
    
    # 1. Muestreo balanceado de imágenes
    paths_s, y_s = sample_balanced(paths_filtered, labels_filtered, cfg['max_per_class'], cfg['random_state'])

    # 2. Extracción de características (con cache)
    cache_key = (
        cfg['img_size'], cfg['feature_set'],
        tuple(paths_s.tolist())
    )
    if cache_key in _feature_cache:
        X = _feature_cache[cache_key]
    else:
        X = extract_features(paths_s, cfg)
        _feature_cache[cache_key] = X

    # 3. División entrenamiento/prueba
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_s, test_size=cfg['test_size'], stratify=y_s, random_state=cfg['random_state']
    )

    # 4. Entrenamiento
    est = build_estimator(cfg)
    est.fit(X_train, y_train)
    y_pred = est.predict(X_test)

    # 5. Resultados
    print('=' * 60)
    print('CONFIGURACION')
    print('=' * 60)
    print(f"Flores seleccionadas: {', '.join(selected_class_names)}")
    print(f"Características: {cfg['feature_set']}")
    print(f"Modelo: {cfg['model']}")
    print(f"Imágenes de entrenamiento: {len(X_train)}, de prueba: {len(X_test)}")
    print()

    print('=' * 60)
    print('RESULTADOS')
    print('=' * 60)
    print(classification_report(y_test, y_pred, target_names=selected_class_names, digits=3))
    print()

    # 6. Matriz de confusión
    print('Matriz de confusión (que clases se confunden):')
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=selected_class_names)
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
    plt.tight_layout()
    plt.show()

    return est


# Ejecuta un experimento de prueba con la configuración por defecto
print('Ejecutando experimento inicial...')
_ = run_experiment(dict(BASE_CONFIG))

## PASO 6: Panel de Control - Tu laboratorio interactivo

A continuación verás controles interactivos (sliders y menúes) para cambiar los parámetros del experimento en tiempo real.

Recordatorio: en la Unidad 4 aprendiste que cada algoritmo tiene parámetros que pueden mejorar o empeorar su desempeño:
- Para SVM: el parámetro C controla cuánto "rigidez" tiene el modelo (valores altos = modelo más complejo, mayor riesgo de overfitting).
- Para Random Forest: el número de árboles afecta la robustez (más árboles generalmente es mejor, pero más lento).
- Para kNN: el valor k define cuántos vecinos mirar (k pequeño = modelo sensible, k grande = modelo suave).

Aquí está el flujo de experimentación sugerido:

1. **Línea base**: Ejecuta con la configuración por defecto (fusion + SVM). Memoriza el resultado.
2. **Prueba descriptores**: Cambia a "color" solo, luego "textura" solo, luego "forma" solo. ¿Cuál da mejor precisión?
3. **Prueba algoritmos**: Mantén "fusion" pero cambia de SVM a Regresión Logística, Random Forest, y kNN. ¿Cuáles son más rápidos? ¿Cuáles son más precisos?
4. **Ajusta parámetros**: Juega con C (SVM), número de árboles (RF), o k (kNN). ¿El modelo mejora o empeora?
5. **Analiza errores**: Mira la matriz de confusión. ¿Qué flores se confunden? ¿Por qué crees que ocurre?



In [ ]:
# PASO 6: Interfaz interactiva con controles (sliders y menúes)

# Widget para seleccionar flores (multi-select)
flowers_select = widgets.SelectMultiple(
    options=classes,
    value=classes,
    description='Flores:',
    rows=len(classes)
)

# Crear controles para características
feature_dd = widgets.Dropdown(
    options=['color','textura','forma','fusion'], 
    value=BASE_CONFIG['feature_set'], 
    description='Características:'
)
model_dd = widgets.Dropdown(
    options=['logreg','svm','rf','knn'], 
    value=BASE_CONFIG['model'], 
    description='Modelo:'
)

# Controles de datos
maxpc = widgets.IntSlider(value=BASE_CONFIG['max_per_class'], min=30, max=250, step=10, 
                          description='Imágenes/clase')
imgsz = widgets.IntSlider(value=BASE_CONFIG['img_size'], min=64, max=160, step=16, 
                          description='Tamaño imagen')

# Controles de parámetros de modelos
svm_c = widgets.FloatLogSlider(value=BASE_CONFIG['svm_c'], base=10, min=-1, max=1.0, step=0.05, 
                               description='SVM (C)')
rf_trees = widgets.IntSlider(value=BASE_CONFIG['rf_trees'], min=50, max=600, step=50, 
                             description='RF (árboles)')
knn_k = widgets.IntSlider(value=BASE_CONFIG['knn_k'], min=1, max=21, step=2, 
                          description='kNN (vecinos)')

# Botón para ejecutar
btn = widgets.Button(description='Ejecutar experimento', button_style='success')
out = widgets.Output()

param_box = widgets.VBox([svm_c, rf_trees, knn_k])

# Mostrar/ocultar parámetros según el modelo elegido
def refresh_param_visibility(*_):
    svm_c.layout.display = 'none'
    rf_trees.layout.display = 'none'
    knn_k.layout.display = 'none'

    if model_dd.value == 'svm':
        svm_c.layout.display = ''
    elif model_dd.value == 'rf':
        rf_trees.layout.display = ''
    elif model_dd.value == 'knn':
        knn_k.layout.display = ''

refresh_param_visibility()
model_dd.observe(refresh_param_visibility, names='value')


# Función que se ejecuta cuando presionas el botón
def on_run(_):
    with out:
        out.clear_output(wait=True)
        cfg = dict(BASE_CONFIG)
        cfg['feature_set'] = feature_dd.value
        cfg['model'] = model_dd.value
        cfg['max_per_class'] = int(maxpc.value)
        cfg['img_size'] = int(imgsz.value)
        cfg['svm_c'] = float(svm_c.value)
        cfg['rf_trees'] = int(rf_trees.value)
        cfg['knn_k'] = int(knn_k.value)

        selected = list(flowers_select.value) if flowers_select.value else list(classes)
        _ = run_experiment(cfg, selected_classes=selected)

btn.on_click(on_run)

# Armamos la interfaz
print('=' * 60)
print('PANEL DE CONTROL')
print('=' * 60)
ui = widgets.VBox([
    widgets.HTML('<b>Selecciona flores a clasificar (click para elegir/deseleccionar):</b>'),
    flowers_select,
    
    widgets.HTML('<b>Selecciona características y modelo:</b>'),
    widgets.HBox([feature_dd, model_dd]),
    
    widgets.HTML('<b>Controles de datos:</b>'),
    widgets.HBox([maxpc, imgsz]),
    
    widgets.HTML('<b>Parámetros del modelo seleccionado:</b>'),
    param_box,
    
    btn,
    out
])
ui

---

## Reflexión Final y Recursos para Aprender Más

Recordatorio: lo que has hecho aquí es implementar un **pipeline completo de aprendizaje automático**:
- **Etapa 1**: Recolección y preparación de datos (descargaste flores, las redimensionaste).
- **Etapa 2**: Extracción de características (aplicaste descriptores clásicos).
- **Etapa 3**: Entrenamiento (usaste un algoritmo para aprender patrones).
- **Etapa 4**: Evaluación (mediste precisión, recall, y miraste la matriz de confusión).

Este es el mismo flujo que se usa en proyectos reales de visión por computadora, solo que con técnicas clásicas en lugar de redes profundas (que verías en cursos más avanzados).

Si quieres profundizar, aquí hay ideas para extender este trabajo:
- **Reducción de dimensionalidad**: Agrega PCA (Análisis de Componentes Principales) antes de entrenar. ¿Mejora o empeora el desempeño?
- **Validación cruzada**: En lugar de un solo split 75/25, divide los datos en k folds y promedia los resultados.
- **Tu propio dataset**: Descarga imágenes propias (flores locales, mascotas, plantas) y entrena un modelo.
- **Comparación con Deep Learning**: Usa un modelo preentrenado de TensorFlow/Keras y compara resultados.

Notas importantes mientras experimentas:
- Cada experimento puede tomar 30-60 segundos dependiendo del tamaño y el modelo.
- La precisión general es importante, pero también mira Precision y Recall por clase (leyendo el reporte detallado).
- La matriz de confusión es tu mejor aliada para entender qué clases se confunden y por qué.
- Si el accuracy en entrenamiento es mucho mayor que en prueba, probablemente estés en overfitting (el modelo memorizó en lugar de generalizar).

¡Diviértete experimentando y descubriendo cuál combinación funciona mejor!